# 01. Coleta de dados: dengue no Nordeste (SINAN via API) + população (IBGE/SIDRA)

**Checkpoint 1 (CP1).** Exigência: no mínimo 2 fontes integradas e dicionário de dados.

Ordem de execução: rode as células de cima para baixo. As seções 1 a 3 são **diagnóstico**:
elas confirmam o que a API realmente aceita antes de gastar horas coletando.

| Fonte | O que traz | Chave de junção |
|---|---|---|
| API Dados Abertos / Saúde (SINAN) | notificações de dengue | município de residência (6 dígitos) |
| API SIDRA / IBGE | população por município e ano | código do município (7 dígitos, usar os 6 primeiros) |

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
import coleta_dengue_api as dengue_api
import coleta_ibge as ibge
import dicionario
from config import DATA_RAW, DATA_PROCESSED, DOCS, UFS_NORDESTE, ANO_INICIO, ANO_FIM, N_MUNICIPIOS_NE
pd.set_option("display.max_columns", 50)

## 1. Diagnóstico da API de dengue
Mostra o formato da resposta, os campos e a latência. **Guarde a saída** (print) para o relatório.
Se der erro de parâmetro, abra o Swagger e ajuste as constantes no topo de `src/coleta_dengue_api.py`.

In [ ]:
amostra = dengue_api.probe(page_size=5)

## 2. Os filtros funcionam?
Se a API ignora o filtro de ano ou UF, a coleta traria o país inteiro. O teste abaixo diz quanto dos registros
bate com o filtro pedido (esperado: perto de 100%). Ajuste o **nome do campo na resposta** (3º argumento)
conforme os campos vistos na seção 1 (ex.: `nu_ano`, `sg_uf`, `sg_uf_not`).

In [ ]:
dengue_api.testar_filtro(dengue_api.PARAM_ANO, 2023, "nu_ano")
dengue_api.testar_filtro(dengue_api.PARAM_UF, "RN", "sg_uf")

## 3. Quanto vai demorar?
Preencha uma estimativa de linhas (a API pode não informar total). Se o resultado passar de algumas horas,
**a API não é viável para 2000 a 2025**: veja o plano B no README (microdados do DATASUS) e converse com o grupo
antes de continuar.

In [ ]:
# Ajuste o n de linhas com base em um ano piloto (secao 4) e extrapole.
print(dengue_api.estimar_requisicoes(n_linhas=1_000_000, page_size=dengue_api.PAGE_SIZE, seg_por_req=0.5))

## 4. Partição piloto
Baixa poucas páginas de um ano/UF para validar o pipeline ponta a ponta.

In [ ]:
sessao = dengue_api._sessao()
arq = dengue_api.coletar_particao(sessao, ano=2023, uf="RN", max_paginas=3)
piloto = pd.read_json(arq, lines=True, dtype=False)
print(piloto.shape)
piloto.head()

## 5. Coleta completa (dengue)
Fica desligada por padrão. Ligue só depois das seções 1 a 4 fazerem sentido.
É **retomável**: se cair, rode de novo. Dica: divida os anos entre as 3 pessoas do grupo, mas
use pastas de saída separadas e junte depois, para não haver conflito no Git (`data/` fica fora do repositório).

In [ ]:
RODAR_COLETA_COMPLETA = False
ANOS = range(ANO_INICIO, ANO_FIM + 1)

if RODAR_COLETA_COMPLETA:
    dengue_api.coletar(ANOS, list(UFS_NORDESTE))

## 6. População por município (IBGE/SIDRA)
Junta estimativas anuais e Censos 2000/2010/2022. Confira o relatório de **cobertura**:
o esperado são 1794 municípios em cada ano. Anos faltantes (comum nos anos de Censo) serão tratados
por interpolação no notebook de limpeza.

In [ ]:
RODAR_IBGE = True
if RODAR_IBGE:
    ibge.main()
pop = pd.read_csv(DATA_RAW / "ibge" / "populacao_municipios_ne.csv", dtype={"cod_mun7": str, "cod_mun6": str})
pop.head()

## 7. Carregar notificações, filtrar Nordeste e montar o dicionário de dados
O filtro usa o código de município de residência (2 primeiros dígitos entre 21 e 29).
**Ajuste `CAMPO_MUN`** para o nome real visto na seção 1 (ex.: `id_mn_resi`).

In [ ]:
CAMPO_MUN = "id_mn_resi"   # AJUSTAR conforme os campos da API

arquivos = sorted((DATA_RAW / "dengue_api").rglob("*.jsonl")) or sorted((DATA_RAW / "dengue_api").rglob("*.part"))
print(len(arquivos), "arquivos")
df = pd.concat((pd.read_json(a, lines=True, dtype=False) for a in arquivos), ignore_index=True)
df[CAMPO_MUN] = df[CAMPO_MUN].astype(str).str.zfill(6)
df = df[df[CAMPO_MUN].str[:2].astype(int).isin(UFS_NORDESTE.values())]
print(df.shape)

perfil = dicionario.perfil_colunas(df)
perfil

In [ ]:
dicionario.salvar_markdown(perfil, DOCS / "dicionario_dengue_auto.md",
    titulo="Dicionário: notificações de dengue (SINAN, via API Dados Abertos)",
    fonte="https://apidadosabertos.saude.gov.br/v1/")
# Depois, revise o arquivo, preencha os 'PREENCHER' e consolide em docs/dicionario_dados.md

## 8. Integração das duas fontes (teste de junção)
A chave é o código de município de **6 dígitos** (o IBGE usa 7, com dígito verificador).
Aqui só verificamos se a junção funciona e quantos registros ficam sem população; a incidência por 100 mil
será calculada na limpeza, depois de tratar anos sem estimativa.

In [ ]:
ANO_CAMPO = "nu_ano"   # AJUSTAR
df["ano"] = pd.to_numeric(df[ANO_CAMPO], errors="coerce")

casos = (df.groupby([CAMPO_MUN, "ano"]).size().rename("casos_notificados").reset_index()
           .rename(columns={CAMPO_MUN: "cod_mun6"}))
base = casos.merge(pop[["cod_mun6", "ano", "populacao"]], on=["cod_mun6", "ano"], how="left")

print("Linhas município-ano:", len(base))
print("Sem população:", base["populacao"].isna().sum(), f"({base['populacao'].isna().mean():.1%})")
base["incidencia_100mil"] = base["casos_notificados"] / base["populacao"] * 100_000
base.sort_values("incidencia_100mil", ascending=False).head(10)

## 9. Checklist do CP1
- [ ] Seções 1 a 3 executadas e saídas guardadas
- [ ] Pelo menos 1 ano piloto coletado de ponta a ponta
- [ ] Cobertura do IBGE conferida (1794 municípios)
- [ ] Junção dengue x população funcionando
- [ ] `docs/dicionario_dados.md` preenchido (sem "PREENCHER")
- [ ] `docs/uso_ia.md` atualizado
- [ ] Repositório com README, requirements.txt e commits de cada integrante